# 阶段 1：复势构造与解析性验证

目标：构造满足圆柱边界不可穿透条件的复势，并验证在 $r>a$ 区域的解析性。

设总复势由均匀流与偶极子叠加：

$$
\Phi(z)=\Phi_u(z)+\Phi_d(z),\quad \Phi_u(z)=Uz,\quad \Phi_d(z)=\frac{\kappa}{2\pi z}
$$

由边界条件（圆柱表面法向速度为零）

$$
u_r=\frac{\partial \phi}{\partial r}=U\left(1-\frac{a^2}{r^2}\right)\cos\theta,\quad u_r\big|_{r=a}=0
$$

得偶极子强度

$$
\kappa=2\pi Ua^2
$$

因此最终复势公式为

$$
\Phi(z)=U\left(z+\frac{a^2}{z}\right)
$$

接下来分别做：

1. 物理边界条件验证（$u_r(a,\theta)=0$）

2. 极坐标 C-R 方程验证

3. SymPy 符号 + 数值验证（随机点残差）

In [13]:
import platform

import matplotlib

import numpy

import pandas

import sympy



print('Python :', platform.python_version())

print('numpy  :', numpy.__version__)

print('pandas :', pandas.__version__)

print('sympy  :', sympy.__version__)

print('matplotlib:', matplotlib.__version__)


Python : 3.11.15
numpy  : 2.4.3
pandas : 3.0.1
sympy  : 1.14.0
matplotlib: 3.10.8


In [ ]:
from pathlib import Path

import sys


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise RuntimeError(
        "Cannot locate project root. Please open a folder inside flow_simulation_project."
    )


ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root detected.")

Project root detected.


In [9]:
import numpy as np

from src.core.potential import doublet_strength, cylinder_flow



U = 1.5

a = 0.8

kappa = doublet_strength(U, a)



print(f"推导得到 kappa = 2*pi*U*a^2 = {kappa:.6f}")



# 在圆柱表面 r=a 上检查法向速度 u_r 是否为 0

theta = np.linspace(0.0, 2.0 * np.pi, 720, endpoint=False)

ur = U * (1.0 - (a**2) / (a**2)) * np.cos(theta)

print("max|u_r(r=a,theta)| =", np.max(np.abs(ur)))



# 采样几个点查看复势数值

z = np.array([1.2 + 0.5j, -1.5 + 0.7j, 0.9 - 1.1j])

W = cylinder_flow(z, U=U, a=a)

W


推导得到 kappa = 2*pi*U*a^2 = 6.031858
max|u_r(r=a,theta)| = 0.0


array([ 2.4816568 +0.46597633j, -2.77554745+0.80474453j,
        1.77772277-1.12722772j])

In [10]:
import sympy as sp

from src.utils.cr_verify import symbolic_cr_residuals, random_point_cr_check



sym = symbolic_cr_residuals(U=U, a=a)

print("residual_1 =", sp.simplify(sym["residual_1"]))

print("residual_2 =", sp.simplify(sym["residual_2"]))



check = random_point_cr_check(U=U, a=a, n_samples=500, seed=2026)

check


residual_1 = 0
residual_2 = 0


{'n_samples': 500.0,
 'max_abs_r1': 0.0,
 'max_abs_r2': 0.0,
 'mean_abs_r1': 0.0,
 'mean_abs_r2': 0.0}

In [12]:
from src import FlowSimulator

import src.utils.cr_verify as cr

import importlib

import numpy as np



importlib.reload(cr)

verify_cr = cr.verify_cr



sim = FlowSimulator(U=U, a=a, n=180, x_min=-2.5, x_max=2.5, y_min=-2.5, y_max=2.5)

res = sim.run()



dx = (sim.x_max - sim.x_min) / (sim.n - 1)

dy = (sim.y_max - sim.y_min) / (sim.n - 1)



# 只在物理有效域 r>a 上评估数值 C-R 残差

R = np.sqrt(res["X"]**2 + res["Y"]**2)

mask = R > (1.1 * a)

fd_stats = verify_cr(res["phi"], res["psi"], dx=dx, dy=dy, mask=mask)

fd_stats


{'max_abs_r1': 0.0024431137832507943,
 'max_abs_r2': 0.0024398663578546476,
 'mean_abs_r1': 0.00016956815008171484,
 'mean_abs_r2': 0.00017337759260301096}